# 01 – Data Collection: NA Top 100 Ranked Matches

Pulls the **most recent 10 ranked solo/duo matches** (`MATCHES_PER_PLAYER` in `src/config.py`) for the **top 100 NA players** by LP.

| Step | Endpoint | Routing |
|---|---|---|
| 1. Top 100 players | `league-v4 /challengerleagues/by-queue/RANKED_SOLO_5x5` | `na1` |
| 2. Match IDs per player | `match-v5 /matches/by-puuid/{puuid}/ids` | `americas` |
| 3. Match details | `match-v5 /matches/{matchId}` | `americas` |

> **Runtime:** 1 + 100 + up to 1,000 requests (one per unique match). A dev key allows 100 requests / 2 min, so this takes **~20 min**.  
> Every response is cached to `data/raw/`, so the notebook can be stopped and re-run safely; it resumes where it left off.

## Setup

In [ ]:
import sys, logging
from pathlib import Path

import pandas as pd
from tqdm.auto import tqdm

sys.path.append(str(Path.cwd().parent))  # make `src` importable from notebooks/
from src.config import TOP_N, MATCHES_PER_PLAYER, PROCESSED_DIR
from src.riot_api import get_challenger_league, get_match_ids, get_match

logging.basicConfig(level=logging.WARNING, format="%(levelname)s | %(message)s")

## 1. Top 100 Players
The Challenger ladder holds the top 300 NA players. Sort by LP and keep the top 100.  
`refresh=True` pulls a fresh ladder instead of using the cached one.

In [ ]:
league = get_challenger_league(refresh=False)

players = (
    pd.DataFrame(league["entries"])
      .sort_values("leaguePoints", ascending=False)
      .head(TOP_N)
      .reset_index(drop=True)
)
players.insert(0, "ladder_rank", players.index + 1)
players["games"] = players["wins"] + players["losses"]
players["win_rate"] = players["wins"] / players["games"]

print(f"{len(players)} players | LP range: {players.leaguePoints.min()}–{players.leaguePoints.max()}")
players.head()

## 2. Recent Match IDs
One request per player for their last `MATCHES_PER_PLAYER` ranked solo matches (`queue=420`).  
Top players often share games, so match IDs are de-duplicated before fetching details.

In [ ]:
match_ids = {}   # puuid -> list of match IDs
failed_players = []

for puuid in tqdm(players["puuid"], desc="Match IDs"):
    try:
        match_ids[puuid] = get_match_ids(puuid, count=MATCHES_PER_PLAYER)
    except (RuntimeError, PermissionError) as e:
        logging.error("Skipping %s: %s", puuid, e)
        failed_players.append(puuid)

unique_ids = sorted({m for ids in match_ids.values() for m in ids})
total_ids = sum(len(ids) for ids in match_ids.values())
print(f"{total_ids} player-matches -> {len(unique_ids)} unique matches | failed players: {len(failed_players)}")

## 3. Match Details
Fetch each unique match once. Failures are logged and skipped so one bad request doesn't stop the run.  
Re-run this cell to retry any `failed_matches`.

In [ ]:
matches = {}
failed_matches = []

for mid in tqdm(unique_ids, desc="Matches"):
    try:
        data = get_match(mid)
        if data is not None:
            matches[mid] = data
    except PermissionError:
        raise  # expired key: stop and regenerate
    except RuntimeError as e:
        logging.error("Skipping %s: %s", mid, e)
        failed_matches.append(mid)

print(f"Fetched {len(matches)}/{len(unique_ids)} matches | failed: {len(failed_matches)}")

## 4. Build Player-Match Table
One row per **(top-100 player, match)**, built from that player's entry in the match's `participants`.  
Remakes (early surrender / under 5 min) are flagged, not dropped.

In [ ]:
STAT_COLS = [
    "riotIdGameName", "riotIdTagline", "championName", "teamPosition", "teamId", "win",
    "kills", "deaths", "assists", "totalMinionsKilled", "neutralMinionsKilled",
    "goldEarned", "totalDamageDealtToChampions", "damageDealtToObjectives",
    "visionScore", "wardsPlaced", "wardsKilled", "champLevel", "timeCCingOthers",
    "firstBloodKill", "turretKills", "gameEndedInEarlySurrender",
]

def player_match_row(puuid: str, match: dict) -> dict | None:
    """Extract one player's stats + match metadata from a match-v5 payload."""
    info = match["info"]
    p = next((x for x in info["participants"] if x["puuid"] == puuid), None)
    if p is None:
        return None
    row = {
        "matchId": match["metadata"]["matchId"],
        "puuid": puuid,
        "gameCreation": pd.to_datetime(info["gameCreation"], unit="ms"),
        "gameDuration_min": info["gameDuration"] / 60,
        "gameVersion": info["gameVersion"],
        "patch": ".".join(info["gameVersion"].split(".")[:2]),
    }
    row.update({c: p.get(c) for c in STAT_COLS})
    return row

rows = [
    r for puuid, ids in match_ids.items() for mid in ids
    if mid in matches and (r := player_match_row(puuid, matches[mid])) is not None
]
df = pd.DataFrame(rows)

# Derived features
df["cs"] = df["totalMinionsKilled"] + df["neutralMinionsKilled"]
df["cs_per_min"] = df["cs"] / df["gameDuration_min"]
df["gold_per_min"] = df["goldEarned"] / df["gameDuration_min"]
df["kda"] = (df["kills"] + df["assists"]) / df["deaths"].clip(lower=1)
df["is_remake"] = df["gameEndedInEarlySurrender"] | (df["gameDuration_min"] < 5)

print(df.shape)
df.head()

### Add Riot IDs to the player table
The league endpoint doesn't return names, so take each player's most recent Riot ID from their matches.

In [ ]:
latest_names = (
    df.sort_values("gameCreation")
      .groupby("puuid")[["riotIdGameName", "riotIdTagline"]]
      .last()
)
players = players.merge(latest_names, on="puuid", how="left")
players[["ladder_rank", "riotIdGameName", "leaguePoints", "wins", "losses", "win_rate"]].head(10)

## 5. Sanity Checks & Save

In [ ]:
print("Matches per player:\n", df.groupby("puuid").size().describe().round(1), sep="")
print(f"\nRemakes: {df.is_remake.sum()} | Date range: {df.gameCreation.min():%Y-%m-%d} -> {df.gameCreation.max():%Y-%m-%d}")
print(f"Patches: {sorted(df.patch.unique())}")

players.to_parquet(PROCESSED_DIR / "top100_players.parquet", index=False)
df.to_parquet(PROCESSED_DIR / "player_matches.parquet", index=False)
print(f"\nSaved to {PROCESSED_DIR}")